> **Research record.** Cleaned copy of `900-features-stat-areas/120-create-panel-data-quarterly.ipynb` (data checks, dead ends and unlabelled plots removed; outputs cleared).
> Panel of average remote work by statistical area (2011 boundaries) and quarter, with lagged remote work and time-invariant neighbourhood features.
>
> Requires the authors' BigQuery project and the full GPS dataset, which are not public; the notebook is kept as a record of the analysis behind the paper. Run it with its own folder as the working directory and `settings.ini` at the repository root. For a runnable walk-through on synthetic data see `demo/`.

In [ ]:
import geopandas as gpd
from geofunctions import utils
import pandas as pd

## Goal
Build panel data for regression. The spatial unit is statistical area from 2011

In [ ]:
DATE = "20250609"

In [ ]:
QUARTERLY_DATASET_TABLE = f"phd_datasets.sa_dataset_quarterly_{DATE}"

In [ ]:
target_cols = [
    "num_homes",
    "wfh_mean",
    "share_remote_work_days",
    "num_homes_vicinity",
    "share_remote_work_and_unident_days",
]

### Updated with commuting distance on 29/07

In [ ]:
df_rw_2011 = utils.read_query_to_dataframe(
    f"""
with r as (
select  coalesce(yishuv_stat11,cast(sa.yishuv_stat_2022 as string)) yishuv_stat11, sa.yishuv_stat_2022, qdf.quarter,
ST_DISTANCE(
    ST_CENTROID(sa.geometry),
    ST_GEOGPOINT(34.79188915373317, 32.07472355010219)
  ) AS distance_to_azrieli,
  neighborhood_type,
  amenities_1km_area_wght,
  commuting_distance, 
  commuting_distance_wgt_days,
{', '.join(target_cols)},
from mapinamo-2.isr_geo.statistical_areas_2022 sa
left join phd-habidatum.phd_adm.match_census_2022_2011 b
on cast(yishuv_stat_2022 as string)=b.yeshuv_stat_2022
join  {QUARTERLY_DATASET_TABLE} qdf 
on cast(qdf.yishuv_stat_2022 as string)=cast(sa.yishuv_stat_2022 as string)
)
select *  from r
"""
)

In [ ]:
df_rw_2011_agg = (
    df_rw_2011.groupby(["yishuv_stat11", "quarter"])
    .apply(lambda g: (g["wfh_mean"] * g["num_homes"]).sum() / g["num_homes"].sum())
    .reset_index(name="wfh_mean_weighted")
)

In [ ]:
df_rw_2011_agg = df_rw_2011_agg.merge(
    df_rw_2011.groupby(["yishuv_stat11", "quarter"])
    .apply(
        lambda g: (g["share_remote_work_days"] * g["num_homes"]).sum()
        / g["num_homes"].sum()
    )
    .reset_index(name="wfh_mean_vicinity_weighted")
).merge(
    df_rw_2011.groupby(["yishuv_stat11", "quarter"])
    .apply(
        lambda g: (g["share_remote_work_and_unident_days"] * g["num_homes"]).sum()
        / g["num_homes"].sum()
    )
    .reset_index(name="wfh_mean2_vicinity_weighted")
)

In [ ]:
df_rw_2011_agg2 = (
        df_rw_2011.groupby(["yishuv_stat11", "quarter"])[
            ["num_homes", "num_homes_vicinity"]
        ].sum()
    
)

In [ ]:
df_rw_2011_agg = df_rw_2011_agg.merge(df_rw_2011_agg2, on=["yishuv_stat11", "quarter"])

### Make sure distance, amenities and type of neighborhood are constant

In [ ]:
df_rw_2011['r'] = df_rw_2011.groupby(["yishuv_stat11",'quarter'])['distance_to_azrieli'].rank(method='first')

In [ ]:
const_cols = ["yishuv_stat11", "neighborhood_type",'distance_to_azrieli','amenities_1km_area_wght','commuting_distance', 'commuting_distance_wgt_days']

In [ ]:
df_fixed = df_rw_2011[(df_rw_2011['quarter']=='2020Q1')&(df_rw_2011['r']==1)][const_cols]

In [ ]:
df_rw_2011_agg = df_rw_2011_agg.merge(df_fixed , on=["yishuv_stat11"], how='inner')

## Create panel data

In [ ]:
df_rw_2011_agg["quarter_lag1"] = (
    pd.PeriodIndex(df_rw_2011_agg["quarter"], freq="Q") + 1
).astype(str)
df_rw_2011_agg["quarter_lag2"] = (
    pd.PeriodIndex(df_rw_2011_agg["quarter"], freq="Q") + 2
).astype(str)
df_rw_2011_agg["quarter_lag4"] = (
    pd.PeriodIndex(df_rw_2011_agg["quarter"], freq="Q") + 4
).astype(str)

In [ ]:
target_cols = [
    "wfh_mean_weighted",
    "wfh_mean_vicinity_weighted",
    "wfh_mean2_vicinity_weighted",
]

In [ ]:
df_rw_2011_agg_q1 = (
    df_rw_2011_agg.groupby(["yishuv_stat11", "quarter_lag1"])[target_cols]
    .agg("mean")
    .reset_index()
)
df_rw_2011_agg_q2 = (
    df_rw_2011_agg.groupby(["yishuv_stat11", "quarter_lag2"])[target_cols]
    .agg("mean")
    .reset_index()
)

df_rw_2011_agg_q4 = (
    df_rw_2011_agg.groupby(["yishuv_stat11", "quarter_lag4"])[target_cols]
    .agg("mean")
    .reset_index()
)

## Match

In [ ]:
df_merged = (
    df_rw_2011_agg.merge(
        df_rw_2011_agg_q1,
        left_on=["yishuv_stat11", "quarter"],
        right_on=["yishuv_stat11", "quarter_lag1"],
        suffixes=("", "_lag1"),
        how="left",
    )
    .merge(
        df_rw_2011_agg_q2,
        left_on=["yishuv_stat11", "quarter"],
        right_on=["yishuv_stat11", "quarter_lag2"],
        how="left",
        suffixes=("", "_lag2"),
    )
    .merge(
        df_rw_2011_agg_q4,
        left_on=["yishuv_stat11", "quarter"],
        right_on=["yishuv_stat11", "quarter_lag4"],
        how="left",
        suffixes=("", "_lag4"),
    )
    .drop(["quarter_lag1", "quarter_lag4"], axis=1)
)

In [ ]:
panel_yishuv_stat11 = (
    df_merged.groupby(["yishuv_stat11"])["wfh_mean_vicinity_weighted"]
    .apply(lambda x: x.isnull().sum())[
        df_merged.groupby(["yishuv_stat11"])["wfh_mean_vicinity_weighted"].apply(
            lambda x: x.isnull().sum()
        )
        == 0
    ]
    .index
)

In [ ]:
DATE = "20250730"

In [ ]:
utils.upload_table_to_bq(
    df_merged[df_merged.yishuv_stat11.isin(panel_yishuv_stat11)],
    "phd_datasets",
    f"sa2011_wfh_quarter_for_panel_{DATE}",
)

### FOR LAST MAP (27/08/2025)

In [ ]:
df = utils.read_query_to_dataframe("""select s.*, p.geometry from phd_datasets.sa2011_wfh_quarter_for_panel_20250730 s
                                   join phd-habidatum.phd_adm.stat_areas_2011_isr p
                                   on cast(yishuv_stat11 as string)=cast(yishuv_sta as string)""")

In [ ]:
df['quarter_month'] = pd.PeriodIndex(df["quarter"], freq="Q").to_timestamp()


In [ ]:
gdf = utils.make_gdf(df, crs=4326, geometry='geometry')

In [ ]:
gdf.to_file("../data/results/data_quarterly.geojson")